# Depression Detection — Merged CV Experiment
## Repeated Stratified Group K-Fold on DAIC-WOZ + Clasificación Final (combined)

**Design:**
- **Dataset**: DAIC-WOZ + Clasificación Final embeddings, merged into one pool.
- Manifest-free loader (`root.rglob(model_key)`) — scans the folder tree directly, finds every patient.
- Segments from the same patient always stay in the same fold (grouped by `audio_id`, prefixed per dataset to avoid ID collisions).
- SMOTE / BorderlineSMOTE applied **only inside the training fold** — never on validation.
- StandardScaler fitted **only on training fold**, then applied to validation.
- LazyClassifier used to screen all sklearn classifiers per (embedding model × pipeline).
- Repeated **Stratified Group** K-Fold: 5 splits × 10 repeats = 50 evaluations, grouped by `audio_id`.
- Extended metrics: Accuracy, F1 Macro, F1 (depression class), ROC AUC, Balanced Accuracy.
- Confusion matrix + ROC curve saved for the winning configuration.
- Per-patient majority-vote aggregation for the clinically meaningful metric.

In [1]:
# ==============================================================================
# CELL 0 — Imports & paths
# ==============================================================================
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score, roc_auc_score,
    confusion_matrix, roc_curve, auc, classification_report,
)
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from lazypredict.Supervised import LazyClassifier

warnings.filterwarnings('ignore')

# ── Dataset paths ─────────────────────────────────────────────────────────────
EMBEDDINGS_DIRS = [
    # DAIC-WOZ embeddings (new pipeline)
    "/home/ci2dt2-ai/Proyectos/Psiquiatria/embeddings_daic",
    # Clasificación Final embeddings (existing dataset)
    "/home/ci2dt2-ai/Proyectos/Psiquiatria/Clasificación Final/Depresión/embeddings_v2/depresion",
]

MODELS_TO_TEST = [
    "xlsr-300m",
    "xlsr-53",
    "whisper-large-encoder",
    "wav2vec2-large-robust",
    "wavlm-large",
    "hubert-large",
]

N_SPLITS  = 5
N_REPEATS = 1
SEED      = 42

OUTPUT_DIR = Path("merged_outputs")
PLOTS_DIR  = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

def f1_macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

print('Imports OK')
print(f'Embedding sources: {len(EMBEDDINGS_DIRS)}')


Imports OK
Embedding sources: 2


In [2]:
# ==============================================================================
# CELL 1 — Manifest-free folder scanner (merged across datasets)
#
# Scans the folder structure to find all embedding.json files for a given
# model, across every directory in EMBEDDINGS_DIRS, and concatenates them
# into a single pool. root.rglob(model_key) finds every folder named
# exactly model_key anywhere under root, regardless of whether a
# 'condition' subfolder exists — manifest.json is never read.
#
# Supported layouts:
#   NEW (DAIC):  root/{condition}/{class_X}/{model}/{audio_id}/segment_N/embedding.json
#   OLD (Clas.): root/{class_X}/{model}/{audio_id}/segment_N/embedding.json
#
# Patient IDs are prefixed with the dataset index ("ds0_303", "ds1_004_...")
# to avoid cross-dataset ID collisions when grouping folds.
# ==============================================================================

def scan_embeddings(root: Path, model_key: str, ds_idx: int = 0, label: str = ""):
    """
    Walk root and collect all embedding.json files for model_key.
    Returns (X, y, groups) as lists. groups = "ds{idx}_{audio_id}".
    """
    all_X, all_y, all_groups = [], [], []

    model_dirs = [d for d in root.rglob(model_key) if d.is_dir()]
    if not model_dirs:
        print(f"  [WARN] No folder named {model_key!r} found under {root}")
        return all_X, all_y, all_groups

    n_loaded = n_missing = 0

    for model_dir in sorted(model_dirs):
        class_dir   = model_dir.parent
        class_label = class_dir.name
        if not class_label.startswith("class_"):
            continue
        class_id = int(class_label.split("_")[-1])

        for audio_dir in sorted(model_dir.iterdir()):
            if not audio_dir.is_dir():
                continue
            audio_id = audio_dir.name
            group_id = f"ds{ds_idx}_{audio_id}"

            for seg_dir in sorted(audio_dir.iterdir()):
                if not seg_dir.is_dir() or not seg_dir.name.startswith("segment_"):
                    continue
                emb_file = seg_dir / "embedding.json"
                if not emb_file.exists():
                    n_missing += 1
                    continue
                try:
                    with open(emb_file) as f:
                        data = json.load(f)
                    emb = data.get("embedding")
                    if emb is None:
                        n_missing += 1
                        continue
                    all_X.append(np.array(emb, dtype=np.float32))
                    all_y.append(class_id)
                    all_groups.append(group_id)
                    n_loaded += 1
                except Exception:
                    n_missing += 1

    tag = f"[{label}] " if label else ""
    status = f"{n_loaded} segments loaded"
    if n_missing:
        status += f", {n_missing} missing/corrupt"
    print(f"  {tag}ds{ds_idx} ({root.name}): {status}")
    return all_X, all_y, all_groups


def load_merged_embeddings(embeddings_dirs: list, model_key: str):
    """
    Load and merge embeddings for model_key from all directories.

    Returns
    -------
    X      : (N, D) float32  — embedding vectors
    y      : (N,)   int      — binary labels (0 / 1)
    groups : (N,)   str      — patient IDs prefixed by dataset index
    """
    all_X, all_y, all_groups = [], [], []

    for ds_idx, emb_dir in enumerate(embeddings_dirs):
        root = Path(emb_dir)
        if not root.exists():
            print(f"  [WARN] Directory not found: {root}")
            continue
        X_ds, y_ds, g_ds = scan_embeddings(root, model_key, ds_idx=ds_idx)
        all_X.extend(X_ds)
        all_y.extend(y_ds)
        all_groups.extend(g_ds)

    if not all_X:
        raise ValueError(f'No embeddings found for model "{model_key}"')

    X      = np.stack(all_X, axis=0).astype(np.float32)
    y      = np.array(all_y, dtype=np.int32)
    groups = np.array(all_groups)

    print(f"  Merged: X={X.shape}, y={y.shape}, "
          f"class_0={int((y==0).sum())}, class_1={int((y==1).sum())}, "
          f"patients={len(np.unique(groups))}")
    return X, y, groups


# Quick sanity check
print("Testing loader with xlsr-300m ...")
X_test, y_test, g_test = load_merged_embeddings(EMBEDDINGS_DIRS, "xlsr-300m")
print(f"Segments: {len(X_test)}  |  Embedding dim: {X_test.shape[1]}")


Testing loader with xlsr-300m ...
  ds0 (embeddings_daic): 36452 segments loaded
  ds1 (depresion): 5252 segments loaded
  Merged: X=(41704, 1024), y=(41704,), class_0=29174, class_1=12530, patients=268
Segments: 41704  |  Embedding dim: 1024


In [3]:
# ==============================================================================
# CELL 2 — Helpers: SMOTE-safe resampling, extended metrics, plots
# ==============================================================================

def smote_safe(X, y, kind="smote", seed=SEED):
    """Apply SMOTE/BorderlineSMOTE only if the minority class has enough samples."""
    min_count = int(np.bincount(y).min())
    k = max(1, min(5,  min_count - 1))
    m = max(1, min(10, min_count - 1))
    try:
        if kind == "borderline":
            return BorderlineSMOTE(random_state=seed, k_neighbors=k, m_neighbors=m).fit_resample(X, y)
        return SMOTE(random_state=seed, k_neighbors=k).fit_resample(X, y)
    except Exception:
        return X, y


def extended_metrics(y_true, y_pred, y_prob=None) -> dict:
    """Compute the full metric set used across the project."""
    out = {
        "Accuracy":          accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "F1 Macro":          f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "F1 Weighted":       f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1 Dep":            f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0),
    }
    if y_prob is not None:
        try:
            out["ROC AUC"] = roc_auc_score(y_true, y_prob)
        except Exception:
            out["ROC AUC"] = float("nan")
    return out


def plot_confusion_matrix(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Dep.", "Dep."],
                yticklabels=["No Dep.", "Dep."])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_roc_curve(y_true, y_prob, title, path):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="Random")
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

print("Helpers defined.")


Helpers defined.


In [ ]:
# ==============================================================================
# CELL 3 — Main CV experiment (LazyClassifier × 7 pipelines × 6 models)
#
# For each SSL model × each pipeline:
#   1. Load the merged embedding pool (DAIC-WOZ + Clasificación Final)
#   2. Run StratifiedGroupKFold repeated N_REPEATS times (5 splits × 10 repeats
#      = 50 folds total), grouped by patient_id so segments from the same
#      patient never split across train/val. RepeatedStratifiedKFold does NOT
#      support groups, so we repeat StratifiedGroupKFold manually instead.
#   3. Inside each fold: apply preprocessing (scaler, PCA, SMOTE) fitted on
#      train only, then run LazyClassifier (trains every sklearn classifier
#      on train, evaluates on val)
#   4. Average each classifier's metrics across all folds
#
# Pipelines tested
# ----------------
#   P0  Raw embeddings, no SMOTE, class_weight=balanced
#   P1  Raw embeddings + SMOTE
#   P2  StandardScaler + SMOTE
#   P3  Raw embeddings + BorderlineSMOTE
#   P4  StandardScaler + BorderlineSMOTE
#   P5  StandardScaler + PCA(15) + SMOTE
#   P6  StandardScaler + PCA(15) + BorderlineSMOTE
# ==============================================================================

PIPELINES = {
    "P0_raw_balanced":            {"scaler": False, "pca": False, "smote": None},
    "P1_raw_smote":               {"scaler": False, "pca": False, "smote": "smote"},
    "P2_scaler_smote":            {"scaler": True,  "pca": False, "smote": "smote"},
    "P3_raw_borderline":          {"scaler": False, "pca": False, "smote": "borderline"},
    "P4_scaler_borderline":       {"scaler": True,  "pca": False, "smote": "borderline"},
    "P5_scaler_pca15_smote":      {"scaler": True,  "pca": True,  "smote": "smote"},
    "P6_scaler_pca15_borderline": {"scaler": True,  "pca": True,  "smote": "borderline"},
}

all_lazy_results = []   # one row per (model, pipeline, classifier), averaged over folds

for emb_model in MODELS_TO_TEST:
    print(f"\n{'='*80}")
    print(f"  {emb_model}")
    print(f"{'='*80}")

    try:
        X_mat, y_mat, groups = load_merged_embeddings(EMBEDDINGS_DIRS, emb_model)
    except Exception as e:
        print(f"  [ERROR] loading: {e}")
        continue

    X_full = pd.DataFrame(X_mat)
    y_full = pd.Series(y_mat)

    # RepeatedStratifiedKFold does NOT support `groups` — it would silently
    # ignore patient grouping and leak segments across train/val. Instead we
    # manually repeat StratifiedGroupKFold N_REPEATS times with different
    # random seeds, which DOES respect groups (no patient split across folds).
    repeated_splits = []
    for repeat_i in range(N_REPEATS):
        sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED + repeat_i)
        for tr_idx, val_idx in sgkf.split(X_full, y_full, groups=groups):
            repeated_splits.append((tr_idx, val_idx))

    for pipe_name, pipe_cfg in PIPELINES.items():
        fold_rows = []

        for fold, (tr_idx, val_idx) in enumerate(repeated_splits, start=1):
            X_train, y_train = X_full.iloc[tr_idx].to_numpy(), y_full.iloc[tr_idx].to_numpy()
            X_val,   y_val   = X_full.iloc[val_idx].to_numpy(), y_full.iloc[val_idx].to_numpy()

            # 1. StandardScaler — fit on train, transform both
            if pipe_cfg["scaler"]:
                scaler  = StandardScaler()
                X_train = scaler.fit_transform(X_train)
                X_val   = scaler.transform(X_val)

            # 2. PCA — fit on train, transform both
            if pipe_cfg["pca"]:
                pca     = PCA(n_components=15, random_state=SEED)
                X_train = pca.fit_transform(X_train)
                X_val   = pca.transform(X_val)

            # 3. SMOTE / BorderlineSMOTE — train only
            if pipe_cfg["smote"]:
                X_train, y_train = smote_safe(X_train, y_train, kind=pipe_cfg["smote"])

            # 4. LazyClassifier — train on fold-train, evaluate on fold-val
            clf = LazyClassifier(verbose=0, ignore_warnings=True,
                                 custom_metric=f1_macro, predictions=False)
            if pipe_cfg["smote"] is None:
                for _, m in clf.models.items():
                    if hasattr(m, "class_weight"):
                        m.class_weight = "balanced"

            try:
                lazy_res, _ = clf.fit(
                    pd.DataFrame(X_train), pd.DataFrame(X_val),
                    pd.Series(y_train),    pd.Series(y_val),
                )
                lazy_res = lazy_res.reset_index()
                lazy_res.columns = ["Classifier"] + list(lazy_res.columns[1:])
                lazy_res["Fold"] = fold
                fold_rows.append(lazy_res)
            except Exception as e:
                print(f"  [{pipe_name}] fold {fold} LazyClassifier error: {e}")

        if fold_rows:
            df_folds = pd.concat(fold_rows, ignore_index=True)
            metric_cols = [c for c in ["Accuracy", "Balanced Accuracy", "F1 Score", "f1_macro", "ROC AUC"]
                           if c in df_folds.columns]
            avg = (
                df_folds
                .groupby("Classifier", as_index=False)[metric_cols]
                .mean()
            )
            avg["Embedding_Model"] = emb_model
            avg["Pipeline"]        = pipe_name
            all_lazy_results.append(avg)

            top = avg.sort_values("ROC AUC", ascending=False).head(3)
            print(f"\n  [{pipe_name}] top-3 by ROC AUC (mean over {N_SPLITS}x{N_REPEATS} folds):")
            for _, r in top.iterrows():
                print(f"    {r['Classifier']:<40s} roc_auc={r.get('ROC AUC', float('nan')):.3f}  "
                      f"f1_macro={r.get('f1_macro', float('nan')):.3f}  "
                      f"acc={r.get('Accuracy', float('nan')):.3f}")

# ── Aggregate & save ─────────────────────────────────────────────────────────
if all_lazy_results:
    df_all = pd.concat(all_lazy_results, ignore_index=True)
    df_all.to_csv(OUTPUT_DIR / "merged_cv_all_results.csv", index=False)
    print(f"\nAll LazyClassifier results saved → {OUTPUT_DIR / 'merged_cv_all_results.csv'}  "
          f"({len(df_all)} rows)")

    print(f"\n{'='*80}")
    print("GLOBAL TOP-20 (by ROC AUC)")
    print(f"{'='*80}")
    top20 = df_all.sort_values("ROC AUC", ascending=False).head(20)
    cols = [c for c in ["Embedding_Model","Pipeline","Classifier","Accuracy",
                         "Balanced Accuracy","F1 Score","f1_macro","ROC AUC"]
            if c in top20.columns]
    print(top20[cols].to_string(index=False))



  xlsr-300m
  ds0 (embeddings_daic): 36452 segments loaded
  ds1 (depresion): 5252 segments loaded
  Merged: X=(41704, 1024), y=(41704,), class_0=29174, class_1=12530, patients=268


High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.



  [P0_raw_balanced] top-3 by ROC AUC (mean over 5x1 folds):
    NuSVC                                    roc_auc=0.541  f1_macro=0.512  acc=0.654
    SVC                                      roc_auc=0.540  f1_macro=0.486  acc=0.670
    SGDClassifier                            roc_auc=0.539  f1_macro=0.519  acc=0.620


High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.



  [P1_raw_smote] top-3 by ROC AUC (mean over 5x1 folds):
    SVC                                      roc_auc=0.539  f1_macro=0.531  acc=0.601
    NuSVC                                    roc_auc=0.537  f1_macro=0.531  acc=0.608
    Perceptron                               roc_auc=0.528  f1_macro=0.505  acc=0.543


...

High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1024 features). Some models may be slow or fail. Consider dimensionality reduction.



  [P4_scaler_borderline] top-3 by ROC AUC (mean over 5x1 folds):
    BernoulliNB                              roc_auc=0.527  f1_macro=0.505  acc=0.536
    NearestCentroid                          roc_auc=0.521  f1_macro=0.511  acc=0.560
    RandomForestClassifier                   roc_auc=0.514  f1_macro=0.504  acc=0.648

  [P5_scaler_pca15_smote] top-3 by ROC AUC (mean over 5x1 folds):
    CalibratedClassifierCV                   roc_auc=0.511  f1_macro=0.340  acc=0.356
    DecisionTreeClassifier                   roc_auc=0.502  f1_macro=0.497  acc=0.545
    Perceptron                               roc_auc=0.501  f1_macro=0.491  acc=0.530

  [P6_scaler_pca15_borderline] top-3 by ROC AUC (mean over 5x1 folds):
    CalibratedClassifierCV                   roc_auc=0.514  f1_macro=0.345  acc=0.359
    DummyClassifier                          roc_auc=0.500  f1_macro=0.412  acc=0.700
    Perceptron                               roc_auc=0.499  f1_macro=0.487  acc=0.523

  whisper-large-enco

High-dimensional dataset (1280 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1280 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1280 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1280 features). Some models may be slow or fail. Consider dimensionality reduction.
High-dimensional dataset (1280 features). Some models may be slow or fail. Consider dimensionality reduction.



  [P0_raw_balanced] top-3 by ROC AUC (mean over 5x1 folds):
    BernoulliNB                              roc_auc=0.527  f1_macro=0.513  acc=0.548
    AdaBoostClassifier                       roc_auc=0.527  f1_macro=0.469  acc=0.670
    ExtraTreesClassifier                     roc_auc=0.523  f1_macro=0.421  acc=0.693


High-dimensional dataset (1280 features). Some models may be slow or fail. Consider dimensionality reduction.


In [1]:
# ==============================================================================
# CELL 4 — Summary: best pipeline × classifier per embedding model + heatmap
# ==============================================================================

if 'df_all' in dir() and len(df_all):
    best_per_model = (
        df_all
        .sort_values("ROC AUC", ascending=False)
        .groupby("Embedding_Model", as_index=False)
        .first()
    )
    cols = [c for c in ["Embedding_Model","Pipeline","Classifier","Accuracy",
                         "Balanced Accuracy","F1 Score","f1_macro","ROC AUC"]
            if c in best_per_model.columns]
    best_per_model = best_per_model[cols].sort_values("ROC AUC", ascending=False)

    print("Best configuration per SSL model:")
    print(best_per_model.to_string(index=False))
    best_per_model.to_csv(OUTPUT_DIR / "merged_cv_best_per_model.csv", index=False)
    print(f"\nSaved → {OUTPUT_DIR / 'merged_cv_best_per_model.csv'}")

    # Heatmap — best ROC AUC per (embedding_model × pipeline)  [PRIMARY metric]
    if "ROC AUC" in df_all.columns:
        pivot_auc = (
            df_all
            .groupby(["Embedding_Model", "Pipeline"])["ROC AUC"]
            .max()
            .unstack("Pipeline")
            .fillna(0)
        )
        plt.figure(figsize=(14, 5))
        sns.heatmap(pivot_auc, annot=True, fmt=".3f", cmap="YlGnBu",
                    linewidths=0.5, vmin=0.3, vmax=1.0)
        plt.title("Best ROC AUC — Repeated Stratified Group K-Fold (DAIC-WOZ + Clasificación Final)")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "merged_cv_heatmap_auc.png", dpi=150)
        plt.show()
        print(f"Saved → {PLOTS_DIR / 'merged_cv_heatmap_auc.png'}")

    # Secondary heatmap — F1 Macro, for reference
    pivot = (
        df_all
        .groupby(["Embedding_Model", "Pipeline"])["f1_macro"]
        .max()
        .unstack("Pipeline")
        .fillna(0)
    )
    plt.figure(figsize=(14, 5))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd",
                linewidths=0.5, vmin=0.3, vmax=1.0)
    plt.title("Best F1 Macro — Repeated Stratified Group K-Fold (DAIC-WOZ + Clasificación Final)")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "merged_cv_heatmap.png", dpi=150)
    plt.show()
    print(f"Saved → {PLOTS_DIR / 'merged_cv_heatmap.png'}")


In [ ]:
# ==============================================================================
# CELL 5 — Detailed evaluation of the winning configuration
#
# Rebuilds the best (model, pipeline, classifier) combination found in Cell 4.
# Runs a single representative StratifiedKFold pass to produce out-of-fold
# predictions for the WHOLE merged dataset, then computes the extended
# metric set, confusion matrix, and ROC curve from those out-of-fold preds.
#
# Update EMB_MODEL / PIPE_NAME / CLASSIFIER_NAME below from Cell 4's table,
# or leave as-is to auto-pick the global best.
# ==============================================================================
from sklearn.utils import all_estimators
from sklearn.model_selection import StratifiedKFold

# ← Auto-pick the best row from Cell 4, or override manually
if 'best_per_model' in dir() and len(best_per_model):
    best_row = best_per_model.iloc[0]
    EMB_MODEL       = best_row["Embedding_Model"]
    PIPE_NAME       = best_row["Pipeline"]
    CLASSIFIER_NAME = best_row["Classifier"]
else:
    EMB_MODEL       = "wav2vec2-large-robust"
    PIPE_NAME       = "P4_scaler_borderline"
    CLASSIFIER_NAME = "ExtraTreesClassifier"

print(f"Winning configuration: {EMB_MODEL} | {PIPE_NAME} | {CLASSIFIER_NAME}")

pipe_cfg = PIPELINES[PIPE_NAME]
X_mat, y_mat, groups = load_merged_embeddings(EMBEDDINGS_DIRS, EMB_MODEL)

# Find the matching sklearn estimator class by name
estimator_cls = None
for name, cls in all_estimators(type_filter="classifier"):
    if name == CLASSIFIER_NAME:
        estimator_cls = cls
        break
if estimator_cls is None:
    raise ValueError(f"Could not find classifier class for '{CLASSIFIER_NAME}'")

classes = np.unique(y_mat)
y_prob_oof = np.zeros(len(y_mat))
y_pred_oof = np.empty(len(y_mat), dtype=y_mat.dtype)

cv_plot = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for tr_idx, val_idx in cv_plot.split(X_mat, y_mat):
    X_train, y_train = X_mat[tr_idx].copy(), y_mat[tr_idx].copy()
    X_val             = X_mat[val_idx].copy()

    if pipe_cfg["scaler"]:
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)

    if pipe_cfg["pca"]:
        pca     = PCA(n_components=15, random_state=SEED)
        X_train = pca.fit_transform(X_train)
        X_val   = pca.transform(X_val)

    if pipe_cfg["smote"]:
        X_train, y_train = smote_safe(X_train, y_train, kind=pipe_cfg["smote"])

    try:
        model = estimator_cls(random_state=SEED)
    except TypeError:
        model = estimator_cls()
    if hasattr(model, "class_weight") and pipe_cfg["smote"] is None:
        model.set_params(class_weight="balanced")

    model.fit(X_train, y_train)
    y_pred_oof[val_idx] = model.predict(X_val)
    if hasattr(model, "predict_proba"):
        y_prob_oof[val_idx] = model.predict_proba(X_val)[:, 1]

y_prob = y_prob_oof if hasattr(estimator_cls(), "predict_proba") else None
y_pred = y_pred_oof

metrics = extended_metrics(y_mat, y_pred, y_prob)
print(f"\n{'='*60}")
print(f"  OUT-OF-FOLD RESULTS — {EMB_MODEL} | {PIPE_NAME} | {CLASSIFIER_NAME}")
print(f"{'='*60}")
for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.4f}")
print(f"\n{classification_report(y_mat, y_pred, zero_division=0)}")

plot_confusion_matrix(
    y_mat, y_pred,
    title=f"{EMB_MODEL} | {PIPE_NAME} | {CLASSIFIER_NAME}",
    path=PLOTS_DIR / "merged_cv_best_confusion.png",
)
if y_prob is not None:
    plot_roc_curve(
        y_mat, y_prob,
        title=f"ROC: {EMB_MODEL} | {PIPE_NAME} | {CLASSIFIER_NAME}",
        path=PLOTS_DIR / "merged_cv_best_roc.png",
    )

print(f"\nSaved confusion matrix → {PLOTS_DIR / 'merged_cv_best_confusion.png'}")
if y_prob is not None:
    print(f"Saved ROC curve        → {PLOTS_DIR / 'merged_cv_best_roc.png'}")


In [2]:
# ==============================================================================
# CELL 6 — Per-patient majority vote (clinically meaningful metric)
#
# Aggregates segment-level out-of-fold predictions to one prediction per
# patient using majority vote, then recomputes the full metric set at the
# patient level.
# ==============================================================================

df_preds = pd.DataFrame({
    "audio_id":   groups,
    "y_true":     y_mat,
    "y_pred":     y_pred,
})
if y_prob is not None:
    df_preds["y_prob_dep"] = y_prob

agg = {"y_true": ("y_true", "first"),
       "y_pred_mv": ("y_pred", lambda x: int(x.mode()[0]))}
if "y_prob_dep" in df_preds.columns:
    agg["y_prob_mean"] = ("y_prob_dep", "mean")

patient_df = df_preds.groupby("audio_id").agg(**agg).reset_index()

patient_metrics = extended_metrics(
    patient_df["y_true"], patient_df["y_pred_mv"],
    patient_df["y_prob_mean"] if "y_prob_mean" in patient_df.columns else None,
)

print(f"Per-patient majority vote — {EMB_MODEL} | {PIPE_NAME} | {CLASSIFIER_NAME}")
print(f"Patients: {len(patient_df)}")
print(f"{'='*50}")
for k, v in patient_metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

patient_df.to_csv(OUTPUT_DIR / "merged_cv_patient_predictions.csv", index=False)
print(f"\nSaved → {OUTPUT_DIR / 'merged_cv_patient_predictions.csv'}")

if "y_prob_mean" in patient_df.columns:
    plot_confusion_matrix(
        patient_df["y_true"], patient_df["y_pred_mv"],
        title=f"Per-Patient Majority Vote — {EMB_MODEL} | {PIPE_NAME}",
        path=PLOTS_DIR / "merged_cv_patient_confusion.png",
    )
    plot_roc_curve(
        patient_df["y_true"], patient_df["y_prob_mean"],
        title=f"Per-Patient ROC — {EMB_MODEL} | {PIPE_NAME}",
        path=PLOTS_DIR / "merged_cv_patient_roc.png",
    )
    print(f"Saved → {PLOTS_DIR / 'merged_cv_patient_confusion.png'}")
    print(f"Saved → {PLOTS_DIR / 'merged_cv_patient_roc.png'}")


NameError: name 'pd' is not defined